# Intermediate employment spells in O*NET major groups 17 and 19

The introduction of current notebook:

- This notebook creates a broad, downloadable employment-spell-level dataset for later local sample construction.
- It intentionally does **not** restrict industries or detailed occupations, remove internships, collapse records to the user-company level, or restrict the sample to the United States.

The Fabric-stage restrictions are limited to:

1. employment starts in `[2021-01-01, 2024-01-01)`;
2. usable position, user, company, position-number, country, and raw-or-translated-title information; and
3. a delivered O*NET code whose first two characters are `17` or `19`.

The output remains one row per retained source employment spell.

In [ ]:
from pathlib import Path
from zipfile import ZIP_STORED, ZipFile

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, StringType, TimestampType


# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 0. Parameters, export schema, and lightweight helpers
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


POSITION_TABLE = "user_positions"
OUTPUT_NAME = "NewEmpSpells_TwoOccGroups_AllInd"
OUTPUT_ROOT = "Files/WenzhiW/B03_1stRoundFinalNewHires"
OUTPUT_PARQUET = f"{OUTPUT_ROOT}/{OUTPUT_NAME}"
OUTPUT_WRITE_MODE = "overwrite"

COHORT_START_DATE = "2021-01-01"
COHORT_END_DATE_EXCLUSIVE = "2024-01-01"
FOCAL_ONET_MAJOR_GROUPS = ("17", "19")

# These fields are sufficient for the planned local occupation, industry, internship,
# user-company, chronology, and US restrictions. Bulky fields are intentionally omitted.
EXPORT_SOURCE_COLUMNS = (
    "user_id",
    "position_id",
    "position_number",
    "rcid",
    "company_raw",
    "company_cleaned",
    "company_name",
    "startdate",
    "enddate",
    "country",
    "state",
    "location_raw",
    "city",
    "msa",
    "metro_area",
    "title_raw",
    "title_translated",
    "seniority",
    "onet_code",
    "onet_title",
    "job_category",
    "role_k50",
    "role_k150",
    "role_k300",
    "role_k500",
    "role_k1000",
    "role_k1500",
    "naics_code",
    "naics_description",
    "rics_k50",
    "rics_k200",
    "rics_k400",
)

REQUIRED_IDENTIFIER_COLUMNS = (
    "position_id",
    "user_id",
    "rcid",
    "position_number",
)
TEXT_COLUMNS_TO_STANDARDIZE = (
    "country",
    "title_raw",
    "title_translated",
    "onet_code",
)
MISSING_TEXT_VALUES = ("", "empty", "null", "none", "nan", "na", "n/a")

assert len(EXPORT_SOURCE_COLUMNS) == len(set(EXPORT_SOURCE_COLUMNS))
assert set(REQUIRED_IDENTIFIER_COLUMNS).issubset(EXPORT_SOURCE_COLUMNS)
assert set(TEXT_COLUMNS_TO_STANDARDIZE).issubset(EXPORT_SOURCE_COLUMNS)


def clean_text(column_or_name):
    """Trim text and convert delivered missing-value strings to Spark null."""

    column = F.col(column_or_name) if isinstance(column_or_name, str) else column_or_name
    text = F.trim(column.cast("string"))
    is_missing = text.isNull() | F.lower(text).isin(*MISSING_TEXT_VALUES)
    return F.when(is_missing, F.lit(None).cast("string")).otherwise(text)


def standardized_identifier(column_name, data_type):
    """Clean text identifiers without changing the delivered type of numeric IDs."""

    if isinstance(data_type, StringType):
        return clean_text(column_name)
    return F.col(column_name)


def count_true(condition, alias):
    """Count rows satisfying a validation condition, including for an empty sample."""

    return F.coalesce(F.sum(condition.cast("long")), F.lit(0).cast("long")).alias(alias)


def validate_required_columns(data, required_columns, dataset_name):
    """Fail from schema metadata when an expected source column is absent."""

    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise ValueError(f"{dataset_name} is missing required columns: {missing_columns}.")


spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

## Step 1. Define the narrow, early-filtered Spark query

- This step reads only the fields needed for local sample construction.
- It standardizes only fields needed for the Fabric restrictions, derives an exact daily start date, and applies all restrictions before any cache or shuffle.
- The O*NET rule uses the first two trimmed characters of the variable `onet_code`.

In [ ]:

# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 1. Read, standardize restriction fields, and filter employment spells
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


user_positions_source = spark.read.table(POSITION_TABLE)
validate_required_columns(user_positions_source, EXPORT_SOURCE_COLUMNS, POSITION_TABLE)

source_types = {field.name: field.dataType for field in user_positions_source.schema.fields}
startdate_type = source_types["startdate"]
startdate_sql_type = startdate_type.simpleString()

if isinstance(startdate_type, DateType):
    cleaned_start_date = F.col("startdate")
    raw_date_filter = (
        (F.col("startdate") >= F.lit(COHORT_START_DATE).cast("date"))
        & (F.col("startdate") < F.lit(COHORT_END_DATE_EXCLUSIVE).cast("date"))
    )
elif isinstance(startdate_type, TimestampType) or startdate_sql_type.startswith("timestamp"):
    cleaned_start_date = F.col("startdate").cast("date")
    raw_date_filter = (
        (F.col("startdate") >= F.lit(COHORT_START_DATE).cast(startdate_sql_type))
        & (
            F.col("startdate")
            < F.lit(COHORT_END_DATE_EXCLUSIVE).cast(startdate_sql_type)
        )
    )
elif isinstance(startdate_type, StringType):
    cleaned_start_date = F.expr("try_cast(trim(`startdate`) as date)")
    raw_date_filter = (
        (cleaned_start_date >= F.lit(COHORT_START_DATE).cast("date"))
        & (cleaned_start_date < F.lit(COHORT_END_DATE_EXCLUSIVE).cast("date"))
    )
else:
    raise TypeError(
        "startdate must be a date, timestamp, or text field; "
        f"found {startdate_sql_type}."
    )

if not isinstance(source_types["onet_code"], StringType):
    raise TypeError(
        "onet_code must be a text field; "
        f"found {source_types['onet_code'].simpleString()}."
    )

# Apply date and trimmed O*NET-prefix filters before other text standardization.
raw_onet_filter = F.substring(F.trim(F.col("onet_code")), 1, 2).isin(
    *FOCAL_ONET_MAJOR_GROUPS
)
raw_filtered_spells = (
    user_positions_source.select(*EXPORT_SOURCE_COLUMNS)
    .filter(raw_date_filter & raw_onet_filter)
)

standardized_expressions = {}
for column_name in EXPORT_SOURCE_COLUMNS:
    if column_name in TEXT_COLUMNS_TO_STANDARDIZE:
        standardized_expressions[column_name] = clean_text(column_name)
    elif column_name in REQUIRED_IDENTIFIER_COLUMNS:
        standardized_expressions[column_name] = standardized_identifier(
            column_name,
            source_types[column_name],
        )
    else:
        standardized_expressions[column_name] = F.col(column_name)

prepared_spells = raw_filtered_spells.select(
    *[standardized_expressions[name].alias(name) for name in EXPORT_SOURCE_COLUMNS],
    cleaned_start_date.alias("start_date"),
)

has_required_identifiers = F.lit(True)
for column_name in REQUIRED_IDENTIFIER_COLUMNS:
    has_required_identifiers = has_required_identifiers & F.col(column_name).isNotNull()

has_required_country = F.col("country").isNotNull()
has_required_title = (
    F.col("title_raw").isNotNull() | F.col("title_translated").isNotNull()
)
has_valid_start_date = (
    (F.col("start_date") >= F.lit(COHORT_START_DATE).cast("date"))
    & (F.col("start_date") < F.lit(COHORT_END_DATE_EXCLUSIVE).cast("date"))
)
onet_major_group = F.substring(F.col("onet_code"), 1, 2)
has_focal_onet_major_group = onet_major_group.isin(*FOCAL_ONET_MAJOR_GROUPS)

candidate_employment_spells = (
    prepared_spells.filter(
        has_valid_start_date
        & has_required_identifiers
        & has_required_country
        & has_required_title
        & has_focal_onet_major_group
    )
    .select(
        *EXPORT_SOURCE_COLUMNS,
        "start_date",
        F.trunc("start_date", "month").alias("start_month"),
        F.year("start_date").cast("int").alias("start_year"),
        onet_major_group.alias("onet_major_group"),
        F.coalesce("title_translated", "title_raw").alias("title_for_screening"),
    )
)

candidate_employment_spells.explain("formatted")
print(
    f"Validated {POSITION_TABLE}; startdate type is {startdate_sql_type}. "
    "Review the physical plan above for projection and filter pushdown."
)

## Step 2. Materialize once, validate, and summarize

- The filtered dataset is narrow enough to cache for validation, summaries, and writing.
- One aggregation checks the defining restrictions and reports approximate user and company counts without expensive exact distinct shuffles.
- A second aggregation returns only two O*NET major-group rows.

In [ ]:

# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 2. Cache the filtered spells and run compact validations
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


candidate_employment_spells = candidate_employment_spells.persist(StorageLevel.MEMORY_AND_DISK)

invalid_identifier = F.lit(False)
for column_name in REQUIRED_IDENTIFIER_COLUMNS:
    invalid_identifier = invalid_identifier | F.col(column_name).isNull()

invalid_date = (
    F.col("start_date").isNull()
    | (F.col("start_date") < F.lit(COHORT_START_DATE).cast("date"))
    | (F.col("start_date") >= F.lit(COHORT_END_DATE_EXCLUSIVE).cast("date"))
)
invalid_title_or_country = (
    F.col("country").isNull()
    | (F.col("title_raw").isNull() & F.col("title_translated").isNull())
)
invalid_onet_group = ~F.col("onet_major_group").isin(*FOCAL_ONET_MAJOR_GROUPS)

audit = candidate_employment_spells.agg(
    F.count(F.lit(1)).cast("long").alias("employment_spell_count"),
    F.approx_count_distinct("user_id", rsd=0.02).cast("long").alias(
        "approximate_user_count"
    ),
    F.approx_count_distinct("rcid", rsd=0.02).cast("long").alias(
        "approximate_company_count"
    ),
    count_true(invalid_identifier, "invalid_identifier_count"),
    count_true(invalid_date, "invalid_date_count"),
    count_true(invalid_title_or_country, "invalid_title_or_country_count"),
    count_true(invalid_onet_group, "invalid_onet_group_count"),
).first()

validation_fields = (
    "invalid_identifier_count",
    "invalid_date_count",
    "invalid_title_or_country_count",
    "invalid_onet_group_count",
)
validation_errors = [name for name in validation_fields if audit[name] != 0]
if validation_errors:
    candidate_employment_spells.unpersist()
    raise ValueError(f"Final validation failed for: {validation_errors}.")

print("Intermediate employment-spell summary:")
print(f"Employment spells: {audit['employment_spell_count']:,}")
print(f"Approximate users (2% RSD): {audit['approximate_user_count']:,}")
print(f"Approximate companies (2% RSD): {audit['approximate_company_count']:,}")

major_group_summary = (
    candidate_employment_spells.groupBy("onet_major_group")
    .agg(F.count(F.lit(1)).cast("long").alias("employment_spell_count"))
    .orderBy("onet_major_group")
    .collect()
)

print("O*NET major-group counts:")
for row in major_group_summary:
    print(f"Group {row['onet_major_group']}: {row['employment_spell_count']:,} spells")

## Step 3. Write a distributed Parquet dataset

The output remains partitioned so Spark does not incur a global single-file shuffle. Overwrite mode makes reruns deterministic for a fixed source snapshot and parameter set.

In [ ]:

# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 3. Write the employment-spell-level intermediate dataset
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


try:
    candidate_employment_spells.write.mode(OUTPUT_WRITE_MODE).parquet(OUTPUT_PARQUET)
    print(f"Saved multi-part Parquet dataset: {OUTPUT_PARQUET}.")
finally:
    candidate_employment_spells.unpersist()

## Step 4. Create an uncompressed ZIP archive for download

Parquet is already compressed, so the archive stores its files without recompressing them. The ZIP is first written under a temporary name and replaces an older completed archive only after the new archive is successfully closed. This cell is serial driver-side I/O and can be skipped when Fabric provides a convenient directory-download method.

In [ ]:

# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 4. Package the Parquet directory for local download
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


source_directory = Path("/lakehouse/default") / OUTPUT_PARQUET
archive_path = source_directory.parent / f"{OUTPUT_NAME}.zip"
temporary_archive_path = source_directory.parent / f"{OUTPUT_NAME}.zip.incomplete"

if not source_directory.is_dir():
    raise FileNotFoundError(f"Source directory not found: {source_directory}")

source_files = sorted(path for path in source_directory.rglob("*") if path.is_file())
if not source_files:
    raise FileNotFoundError(f"No output files found in: {source_directory}")

if temporary_archive_path.exists():
    temporary_archive_path.unlink()

try:
    with ZipFile(
        temporary_archive_path,
        mode="w",
        compression=ZIP_STORED,
        allowZip64=True,
    ) as archive:
        for source_file in source_files:
            relative_path = source_file.relative_to(source_directory).as_posix()
            archive.write(source_file, arcname=f"{OUTPUT_NAME}/{relative_path}")
    temporary_archive_path.replace(archive_path)
except Exception:
    if temporary_archive_path.exists():
        temporary_archive_path.unlink()
    raise

total_gib = sum(path.stat().st_size for path in source_files) / (1024**3)
print(f"Created: {archive_path}")
print(f"Archived files: {len(source_files):,}")
print(f"Stored Parquet size: {total_gib:,.2f} GiB")